# Experiment J: Top-K Z-score + Focal Loss + Back-Translation Augmentation

Combines all three components from Exp E, Exp B, and Exp H:

- **Top-K z-score** (Exp E): binary indicator of which PCL-discriminative n-grams
  appear; `k ∈ {50, 100, 200}` and `feature_comb_method ∈ {CONCAT, GMF}` searched.
- **Focal Loss** (Exp B/I): down-weights easy negatives; `gamma ∈ {0.5, 1.0, 2.0}` searched.
- **Back-translation augmentation** (Exp H): PCL=1 rows duplicated via EN→pivot→EN
  using pivot languages (RU/FR/FI/DE); `aug_factor ∈ {1, 2, 4}` searched.

The z-score vocabulary and scaler are fit on **original** `train_sub_df` only
(no leakage from augmented rows). The feature factory handles augmented texts
automatically using the same vocabulary. `pos_weight` is recomputed on the
augmented training set and passed to FocalLoss.

Fixed: `POOLING=MEAN`, `warmup_fraction=0.10`, `label_smoothing=0.0`

Searched: `lr`, `weight_decay`, `hidden_dim ∈ {0, 256}`, `dropout_rate ∈ {0.1, 0.3}`,
`head_lr_multiplier ∈ {1, 3, 5}`, `k ∈ {50, 100, 200}`,
`feature_comb_method ∈ {CONCAT, GMF}`, `gamma ∈ {0.5, 1.0, 2.0}`,
`aug_factor ∈ {1, 2, 4}`

In [ ]:
import os
import sys
import random
import logging
import gc
import json

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from transformers import AutoTokenizer
import spacy
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from utils.data import load_data
from utils.split import split_train_val
from utils.dataloaders import make_dataloaders
from utils.pcl_deberta import PCLDeBERTa, PoolingStrategy
from utils.feature_comb import FeatureComb
from utils.fightin_words import compute_fightin_words_zscores, build_topk_ngrams, extract_topk_zscore_features
from utils.losses import FocalLoss
from utils.optim import compute_pos_weight
from utils.training_loop import train_model
from utils.eval import evaluate
from utils.augmentation import backtranslate

SEED = 42
DATA_DIR = "../data"
OUT_DIR = "out"
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256
VAL_FRACTION = 0.15
BATCH_SIZE = 32
N_TRIALS = 20
NUM_EPOCHS = 12
PATIENCE = 4
N_EVAL_STEPS = 35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
POOLING = PoolingStrategy.CLS_MEAN   # fixed: best default for DeBERTa-v3 RTD pretraining
PIVOT_LANGS = ["ru", "fr", "fi", "de"]   # structurally varied, high-quality Helsinki-NLP models

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Data Loading and spaCy Processing

In [ ]:
train_df, dev_df = load_data(DATA_DIR)
train_sub_df, val_sub_df = split_train_val(train_df, val_frac=VAL_FRACTION, seed=SEED)
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

n_pos = int(train_sub_df["binary_label"].sum())
n_neg = len(train_sub_df) - n_pos
LOG.info(f"Train: {len(train_sub_df)} ({n_pos} PCL / {n_neg} non-PCL, "
         f"natural pos frac={n_pos/len(train_sub_df):.3f})")
LOG.info(f"Val: {len(val_sub_df)}, Dev: {len(dev_df)}")

gpu = spacy.prefer_gpu()
nlp = spacy.load("en_core_web_sm")
LOG.info(f"spaCy using {'GPU' if gpu else 'CPU'}")

train_texts = train_sub_df["text"].tolist()
train_docs = list(nlp.pipe(train_texts, batch_size=256))
LOG.info(f"spaCy processed {len(train_docs)} train documents")

## 2. Pre-Compute Back-Translations

Translate all PCL=1 paragraphs in `train_sub_df` through each pivot language.
Results are cached to `data/bt_cache_{lang}.json` — slow only on the first run
(~5–10 min per pivot on GPU). All subsequent runs load from cache instantly.

In [ ]:
pcl_mask = train_sub_df["binary_label"] == 1
pcl_texts = train_sub_df.loc[pcl_mask, "text"].tolist()

bt_texts = {}   # dict[lang -> list[str]]
for lang in PIVOT_LANGS:
    LOG.info(f"Back-translating {len(pcl_texts)} PCL paragraphs via {lang} pivot...")
    bt_texts[lang] = backtranslate(pcl_texts, pivot_lang=lang, device=DEVICE)
LOG.info("All back-translations complete.")

## 3. Compute Z-score Features

Z-score dict and scalers are computed from **original** `train_sub_df` only —
augmented rows are not included here to prevent data leakage.
The feature factory handles augmented texts at trial time using this vocabulary.

In [ ]:
Z_SCORES, _, _, _ = compute_fightin_words_zscores(
    train_docs, train_sub_df["binary_label"].tolist()
)
LOG.info(f"Z-score dictionary: {len(Z_SCORES)} n-grams")

K_VALUES = [50, 100, 200]
topk_ngrams_cache: dict[int, list[str]] = {}
scaler_cache: dict[int, StandardScaler] = {}

for k in K_VALUES:
    topk = build_topk_ngrams(Z_SCORES, k=k)
    topk_ngrams_cache[k] = topk

    train_feats = np.array(
        [extract_topk_zscore_features(doc, topk) for doc in train_docs]
    )
    scaler = StandardScaler()
    scaler.fit(train_feats)
    scaler_cache[k] = scaler
    LOG.info(f"k={k}: top ngrams built, scaler fitted on {train_feats.shape}")

del train_docs
gc.collect()

## 4. Feature Factory

In [ ]:
def make_topk_factory(k: int):
    """Returns an extra_feature_factory for the given k.

    The factory uses the vocabulary and scaler fit on the original train_sub_df,
    so it generalises correctly to back-translated texts without leakage.
    """
    topk = topk_ngrams_cache[k]
    scaler = scaler_cache[k]

    def factory(texts: list[str]) -> torch.Tensor:
        feats = np.array(
            [extract_topk_zscore_features(doc, topk) for doc in nlp.pipe(texts, batch_size=256)]
        )
        scaled = scaler.transform(feats).astype(np.float32)
        return torch.tensor(scaled).to(DEVICE)

    return factory

## 5. Hyperparameter Search

All three components contribute new hyperparameters:
- Z-score: `k ∈ {50, 100, 200}`, `feature_comb_method ∈ {CONCAT, GMF}`
- Focal loss: `gamma ∈ {0.5, 1.0, 2.0}`
- Augmentation: `aug_factor ∈ {1, 2, 4}`

`pos_weight` is recomputed from the augmented training set and passed to
FocalLoss, so both mechanisms jointly address the (reduced) class imbalance.

In [ ]:
EXP_NAME = "J_topk_zscore_focal_aug"


def objective(trial: optuna.trial.Trial) -> float:
    lr             = trial.suggest_float("lr", 4e-6, 6e-5, log=True)
    weight_decay   = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    hidden_dim     = trial.suggest_categorical("hidden_dim", [0, 256])
    dropout_rate   = trial.suggest_categorical("dropout_rate", [0.1, 0.3]) if hidden_dim > 0 else 0.0
    head_lr_mult   = trial.suggest_categorical("head_lr_multiplier", [1, 3, 5])
    k              = trial.suggest_categorical("k", [50, 100, 200])
    feat_comb_name = trial.suggest_categorical("feature_comb_method", ["CONCAT", "GMF"])
    gamma          = trial.suggest_categorical("gamma", [0.5, 1.0, 2.0])
    aug_factor     = trial.suggest_categorical("aug_factor", [1, 2, 4])

    # Fixed — not worth spending trials on
    warmup_fraction = 0.10
    label_smoothing = 0.0

    feat_comb = FeatureComb.CONCAT if feat_comb_name == "CONCAT" else FeatureComb.GMF

    LOG.info(f"[{EXP_NAME}] Trial {trial.number}: lr={lr:.2e}, k={k}, "
             f"feat_comb={feat_comb_name}, gamma={gamma}, aug_factor={aug_factor}")

    # Build augmented training set — different pivot per copy for diversity
    pcl_rows = train_sub_df[train_sub_df["binary_label"] == 1].copy()
    aug_rows = []
    for lang in PIVOT_LANGS[:aug_factor]:   # aug_factor=1→[ru], =2→[ru,fr], =4→all
        aug = pcl_rows.copy()
        aug["text"] = bt_texts[lang]
        aug_rows.append(aug)
    aug_train_df = pd.concat([train_sub_df] + aug_rows, ignore_index=True)

    # pos_weight recomputed on augmented set; passed to FocalLoss
    pos_weight = compute_pos_weight(aug_train_df, DEVICE)
    criterion = FocalLoss(gamma=gamma, pos_weight=pos_weight)

    factory = make_topk_factory(k)
    train_loader, val_loader, dev_loader = make_dataloaders(
        aug_train_df, val_sub_df, dev_df, BATCH_SIZE, MAX_LENGTH, tokeniser, factory
    )

    model = PCLDeBERTa(
        hidden_dim=hidden_dim,
        dropout_rate=dropout_rate,
        n_extra_features=k,
        pooling=POOLING,
        feature_comb_method=feat_comb,
    ).to(DEVICE)

    results = train_model(
        model=model, device=DEVICE,
        train_loader=train_loader, val_loader=val_loader, dev_loader=dev_loader,
        pos_weight=pos_weight, lr=lr, weight_decay=weight_decay,
        num_epochs=NUM_EPOCHS, warmup_fraction=warmup_fraction,
        patience=PATIENCE, head_lr_multiplier=head_lr_mult,
        label_smoothing=label_smoothing, eval_every_n_steps=N_EVAL_STEPS,
        trial=trial, criterion=criterion,
    )

    trial.set_user_attr("best_val_f1",    results["best_val_f1"])
    trial.set_user_attr("best_threshold", results["best_threshold"])
    trial.set_user_attr("dev_f1",         results["dev_metrics"]["f1"])
    trial.set_user_attr("dev_precision",  results["dev_metrics"]["precision"])
    trial.set_user_attr("dev_recall",     results["dev_metrics"]["recall"])

    try:
        prev_best = trial.study.best_value
    except ValueError:
        prev_best = -float("inf")
    if results["best_val_f1"] > prev_best:
        torch.save(
            {k: v.cpu() for k, v in model.state_dict().items()},
            os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_model.pt")
        )
        config = {
            **trial.params,
            "pooling": POOLING.name,
            "warmup_fraction": warmup_fraction,
            "label_smoothing": label_smoothing,
            "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS, "patience": PATIENCE,
            "best_threshold": results["best_threshold"],
        }
        with open(os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_params.json"), "w") as f:
            json.dump(config, f, indent=2)
        LOG.info(f"[{EXP_NAME}] New best saved (val F1={results['best_val_f1']:.4f})")

    del model, train_loader, val_loader, dev_loader, aug_train_df
    gc.collect()
    torch.cuda.empty_cache()
    return results["best_val_f1"]

## 6. Run Experiment

In [ ]:
gc.collect()
torch.cuda.empty_cache()

study = optuna.create_study(
    direction="maximize",
    study_name=f"pcl_deberta_exp_{EXP_NAME}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=300),
)
study.optimize(objective, n_trials=N_TRIALS)

best = study.best_trial
LOG.info(f"Best trial: {best.number}")
LOG.info(f"Val F1: {best.user_attrs['best_val_f1']:.4f} | Dev F1: {best.user_attrs['dev_f1']:.4f}")
LOG.info(f"Best params: {best.params}")

## 7. Results

In [ ]:
for plot_fn, suffix in [
    (plot_optimization_history, "history"),
    (plot_param_importances, "importances"),
    (plot_parallel_coordinate, "parallel"),
]:
    plot_fn(study)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{EXP_NAME}_optuna_{suffix}.png", dpi=300)
    plt.show()

best = study.best_trial
best_params = best.params
feat_comb = FeatureComb.CONCAT if best_params["feature_comb_method"] == "CONCAT" else FeatureComb.GMF
k_best = best_params["k"]

model = PCLDeBERTa(
    hidden_dim=best_params["hidden_dim"],
    dropout_rate=best_params.get("dropout_rate", 0.0),
    n_extra_features=k_best,
    pooling=POOLING,
    feature_comb_method=feat_comb,
).to(DEVICE)

state_dict = torch.load(
    os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_model.pt"), map_location=DEVICE
)
model.load_state_dict(state_dict)

factory = make_topk_factory(k_best)
_, _, dev_loader = make_dataloaders(
    train_sub_df, val_sub_df, dev_df, BATCH_SIZE, MAX_LENGTH, tokeniser, factory
)
dev_metrics = evaluate(model, DEVICE, dev_loader, threshold=best.user_attrs["best_threshold"])

print(f"\n{'='*60}")
print(f"{EXP_NAME.upper()} — Dev Set Results (threshold={best.user_attrs['best_threshold']:.3f})")
print(f"{'='*60}")
print(classification_report(dev_metrics["labels"], dev_metrics["preds"],
                             target_names=["Non-PCL", "PCL"]))
print("Best hyperparams:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"  pooling: mean (fixed)")
print(f"  warmup_fraction: 0.10 (fixed)")
print(f"  label_smoothing: 0.0 (fixed)")

print(f"\nTop-10 PCL n-grams (highest +z):")
for ng in topk_ngrams_cache[k_best][:10]:
    print(f"  {ng!r:30s}  z={Z_SCORES[ng]:+.2f}")

del model
gc.collect()
torch.cuda.empty_cache()